# MirrorSpeech Demo V4 (Colab)

This notebook generates a presentation-ready demo:
- Strictly uses **non-native L2-ARCTIC** test samples
- Auto-extracts L2-ARCTIC zips locally for fast access
- Picks clips with strongest baseline -> full WER improvement
- Outputs:
  - `demo_clips.csv`
  - `demo_metrics.csv`
  - `demo_showcase.html`
  - `clips/` (portable copied audio)

Use **GPU (T4)** runtime.

In [1]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

!pip -q install transformers peft jiwer soundfile torchaudio gdown
!pip install --upgrade torchao

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
# (Optional) Download best checkpoint directly from your shared link id
# Save inside the expected path under MirrorSpeech/checkpoints_balanced10k_27Aprrun
!mkdir -p "/content/drive/MyDrive/MirrorSpeech/checkpoints_balanced10k_27Aprrun"
!gdown "https://drive.google.com/uc?id=1sQE7a6aR-BdvIqiYMWiUnSmZFyQqE73N" -O "/content/drive/MyDrive/MirrorSpeech/checkpoints_balanced10k_27Aprrun/task7_balanced10k_3ep_lr2e5_alpha05_full_best.pt"

Downloading...
From (original): https://drive.google.com/uc?id=1sQE7a6aR-BdvIqiYMWiUnSmZFyQqE73N
From (redirected): https://drive.google.com/uc?id=1sQE7a6aR-BdvIqiYMWiUnSmZFyQqE73N&confirm=t&uuid=6617bb67-bed6-4e9a-8f57-d82a1eac7124
To: /content/drive/MyDrive/MirrorSpeech/checkpoints_balanced10k_27Aprrun/task7_balanced10k_3ep_lr2e5_alpha05_full_best.pt
100% 993M/993M [00:09<00:00, 103MB/s]


In [3]:
# Configure paths / options
# Confirmed valid path from your check output.
DRIVE_ROOT = "/content/drive/MyDrive/MirrorSpeech"
AUTO_DETECT_ROOT = True
BEST_CKPT = "task7_balanced10k_3ep_lr2e5_alpha05_full_best.pt"
NUM_CLIPS = 3
SELECTION_MODE = "best_delta"  # best_delta or balanced_random
CANDIDATE_POOL_PER_ACCENT = 4
OUTPUT_DIR = "/content/demo_artifacts"
TITLE = "MirrorSpeech Before/After Demo"

import glob
import os

if AUTO_DETECT_ROOT:
    candidates = [
        DRIVE_ROOT,
        "/content/drive/MyDrive/MirrorSpeech",
        "/content/drive/MyDrive/Mirrorspeech",
    ]
    candidates.extend(glob.glob("/content/drive/MyDrive/**/MirrorSpeech", recursive=True))
    candidates.extend(glob.glob("/content/drive/MyDrive/**/Mirrorspeech", recursive=True))

    resolved_root = None
    for c in candidates:
        if os.path.isfile(os.path.join(c, "splits", "test.json")) and os.path.isfile(os.path.join(c, "splits", "config.json")):
            resolved_root = c
            break

    if resolved_root is not None:
        DRIVE_ROOT = resolved_root
        print("Using DRIVE_ROOT:", DRIVE_ROOT)
    else:
        print("Could not auto-detect dataset root with splits/test.json.")
        print("Set DRIVE_ROOT manually to your MirrorSpeech folder.")

Using DRIVE_ROOT: /content/drive/MyDrive/MirrorSpeech


In [4]:
# Extract L2-ARCTIC zips from Drive to local /content for fast access.
# This matches the layout used in the Task 4 baseline notebook.
import os
import zipfile

ZIP_DIR = os.path.join(DRIVE_ROOT, "l2arctic_zips")
LOCAL_L2ARCTIC = "/content/data/l2arctic"
os.makedirs(LOCAL_L2ARCTIC, exist_ok=True)

if not os.path.isdir(ZIP_DIR):
    raise FileNotFoundError(
        f"L2-ARCTIC zip folder not found: {ZIP_DIR}\n"
        "Place your speaker zips under <DRIVE_ROOT>/l2arctic_zips/."
    )

zip_files = sorted([f for f in os.listdir(ZIP_DIR) if f.endswith(".zip")])
print(f"Found {len(zip_files)} L2-ARCTIC zips.")

for zip_name in zip_files:
    speaker = zip_name.replace(".zip", "")
    speaker_out = os.path.join(LOCAL_L2ARCTIC, speaker)
    if os.path.isdir(speaker_out) and len(os.listdir(speaker_out)) > 0:
        continue
    zip_path = os.path.join(ZIP_DIR, zip_name)
    print(f"Extracting {zip_name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(LOCAL_L2ARCTIC)

extracted_speakers = sorted([d for d in os.listdir(LOCAL_L2ARCTIC) if os.path.isdir(os.path.join(LOCAL_L2ARCTIC, d))])
print(f"L2-ARCTIC speakers ready: {len(extracted_speakers)}")
print("Sample:", extracted_speakers[:8])

Found 24 L2-ARCTIC zips.
Extracting ABA.zip ...
Extracting ASI.zip ...
Extracting BWC.zip ...
Extracting EBVS.zip ...
Extracting ERMS.zip ...
Extracting HJK.zip ...
Extracting HKK.zip ...
Extracting HQTV.zip ...
Extracting LXC.zip ...
Extracting MBMPS.zip ...
Extracting NCC.zip ...
Extracting NJS.zip ...
Extracting PNV.zip ...
Extracting RRBI.zip ...
Extracting SKA.zip ...
Extracting SVBI.zip ...
Extracting THV.zip ...
Extracting TLV.zip ...
Extracting TNI.zip ...
Extracting TXHC.zip ...
Extracting YBAA.zip ...
Extracting YDCK.zip ...
Extracting YKWK.zip ...
Extracting ZHAA.zip ...
L2-ARCTIC speakers ready: 24
Sample: ['ABA', 'ASI', 'BWC', 'EBVS', 'ERMS', 'HJK', 'HKK', 'HQTV']


In [5]:
import csv
import difflib
import html
import json
import os
import random
import re
import shutil
from dataclasses import dataclass
from typing import Dict, List

import soundfile as sf
import torch
import torchaudio.transforms as T
from jiwer import cer, wer
from peft import LoraConfig, TaskType, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def pick_equally_spaced(rows: List[dict], k: int) -> List[dict]:
    if k >= len(rows):
        return rows
    if k <= 1:
        return [rows[len(rows) // 2]]
    out: List[dict] = []
    step = (len(rows) - 1) / (k - 1)
    for i in range(k):
        out.append(rows[round(i * step)])
    return out

def highlight_prediction(pred: str, gt: str, css_class: str) -> str:
    pred_tokens = pred.split()
    gt_tokens = gt.split()
    matcher = difflib.SequenceMatcher(a=gt_tokens, b=pred_tokens)
    out: List[str] = []
    for tag, _, _, j1, j2 in matcher.get_opcodes():
        segment = pred_tokens[j1:j2]
        if not segment:
            continue
        escaped = " ".join(html.escape(tok) for tok in segment)
        if tag == "equal":
            out.append(escaped)
        else:
            out.append(f'<span class="{css_class}">{escaped}</span>')
    return " ".join(out)

def is_librispeech_record(record: dict) -> bool:
    wav = str(record.get("wav_path", "")).replace("\\", "/").lower()
    speaker = str(record.get("speaker", "")).lower()
    return "librispeech" in wav or speaker.startswith("librispeech")

def resolve_l2arctic_path(record: dict, local_l2_dir: str) -> str:
    return os.path.join(local_l2_dir, record["speaker"], "wav", os.path.basename(str(record["wav_path"]).replace("\\", "/")))

def build_baseline_model(device: torch.device):
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
    model.eval()
    return processor, model

def build_full_model(device: torch.device, ckpt_path: str):
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    model.config.use_cache = False
    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")
    model.generation_config.suppress_tokens = []

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_cfg)

    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["whisper_lora_state"], strict=False)
    model = model.to(device)
    model.eval()
    return processor, model

def transcribe_one(wav_path: str, processor: WhisperProcessor, model: WhisperForConditionalGeneration, device: torch.device, target_sr: int = 16000) -> str:
    waveform, sr = sf.read(wav_path)
    waveform = torch.tensor(waveform, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.T
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != target_sr:
        waveform = T.Resample(sr, target_sr)(waveform)

    audio = waveform.squeeze(0).cpu().numpy()
    feats = processor(audio, sampling_rate=target_sr, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        pred_ids = model.generate(input_features=feats, language="en", task="transcribe")
    pred = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    return pred.strip()

@dataclass
class DemoRow:
    clip_id: str
    accent: str
    audio_file: str
    ground_truth: str
    baseline_pred: str
    full_pred: str
    baseline_wer: float
    full_wer: float
    baseline_cer: float
    full_cer: float

    @property
    def wer_delta(self) -> float:
        return self.baseline_wer - self.full_wer

    @property
    def cer_delta(self) -> float:
        return self.baseline_cer - self.full_cer

def render_html(rows: List[DemoRow], out_html: str, title: str):
    if rows:
        avg_b_wer = sum(r.baseline_wer for r in rows) / len(rows)
        avg_f_wer = sum(r.full_wer for r in rows) / len(rows)
        avg_b_cer = sum(r.baseline_cer for r in rows) / len(rows)
        avg_f_cer = sum(r.full_cer for r in rows) / len(rows)
    else:
        avg_b_wer = avg_f_wer = avg_b_cer = avg_f_cer = 0.0

    cards = []
    for i, r in enumerate(rows, start=1):
        baseline_h = highlight_prediction(r.baseline_pred, r.ground_truth, "wrong")
        full_h = highlight_prediction(r.full_pred, r.ground_truth, "fixed")
        cards.append(f"""
<section class=\"card\">
  <h2>Clip {i}: {html.escape(r.clip_id)} <span class=\"meta\">Accent: {html.escape(r.accent)}</span></h2>
  <audio controls preload=\"metadata\" src=\"{html.escape(r.audio_file)}\"></audio>
  <div class=\"label\">Ground truth</div>
  <div class=\"txt\">{html.escape(r.ground_truth)}</div>
  <div class=\"label\">Baseline (Task 4)</div>
  <div class=\"txt\">{baseline_h}</div>
  <div class=\"label\">Full (Task 7)</div>
  <div class=\"txt\">{full_h}</div>
  <div class=\"metrics\">
    WER: <b>{r.baseline_wer:.3f}</b> -> <b>{r.full_wer:.3f}</b> (delta <b>{r.wer_delta:.3f}</b>)
    &nbsp;|&nbsp;
    CER: <b>{r.baseline_cer:.3f}</b> -> <b>{r.full_cer:.3f}</b> (delta <b>{r.cer_delta:.3f}</b>)
  </div>
</section>
""")

    page = f"""<!doctype html>
<html lang=\"en\">
<head>
<meta charset=\"utf-8\" />
<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
<title>{html.escape(title)}</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 24px; background: #f7f8fa; color: #1f2937; }}
h1 {{ margin-bottom: 16px; }}
.summary {{ background: white; border: 1px solid #e5e7eb; border-radius: 10px; padding: 12px; margin-bottom: 14px; }}
.card {{ background: white; border: 1px solid #e5e7eb; border-radius: 10px; padding: 14px; margin-bottom: 14px; }}
.meta {{ color: #6b7280; font-size: 14px; font-weight: normal; }}
.label {{ margin-top: 8px; font-weight: 700; }}
.txt {{ line-height: 1.45; }}
audio {{ width: 100%; margin-top: 8px; }}
.metrics {{ margin-top: 10px; font-size: 14px; }}
.wrong {{ background: #fde2e1; color: #b42318; padding: 0 2px; border-radius: 3px; }}
.fixed {{ background: #dcfae6; color: #067647; padding: 0 2px; border-radius: 3px; }}
</style>
</head>
<body>
<h1>{html.escape(title)}</h1>
<section class=\"summary\">
  <div><b>Avg WER:</b> {avg_b_wer:.3f} -> {avg_f_wer:.3f} (delta {(avg_b_wer - avg_f_wer):.3f})</div>
  <div><b>Avg CER:</b> {avg_b_cer:.3f} -> {avg_f_cer:.3f} (delta {(avg_b_cer - avg_f_cer):.3f})</div>
  <div><span class=\"wrong\">red</span> = baseline mismatches, <span class=\"fixed\">green</span> = full-model mismatches (vs ground truth)</div>
</section>
{''.join(cards)}
</body>
</html>"""

    with open(out_html, "w", encoding="utf-8") as f:
        f.write(page)

# Paths
splits_dir = os.path.join(DRIVE_ROOT, "splits")
ckpt_dir = os.path.join(DRIVE_ROOT, "checkpoints_balanced10k_27Aprrun")

test_json = os.path.join(splits_dir, "test.json")
config_json = os.path.join(splits_dir, "config.json")
ckpt_path = os.path.join(ckpt_dir, BEST_CKPT)

if not os.path.isfile(test_json):
    raise FileNotFoundError(f"Missing: {test_json}")
if not os.path.isfile(config_json):
    raise FileNotFoundError(f"Missing: {config_json}")
if not os.path.isfile(ckpt_path):
    import glob
    candidates = glob.glob(f"/content/drive/MyDrive/**/{BEST_CKPT}", recursive=True)
    if candidates:
        ckpt_path = candidates[0]
        print("Using fallback checkpoint:", ckpt_path)
    else:
        raise FileNotFoundError(
            f"Missing: {ckpt_path}\nPut the checkpoint under checkpoints_balanced10k_27Aprrun"
        )

with open(config_json, "r", encoding="utf-8") as f:
    config = json.load(f)
accent_map: Dict[int, str] = {int(k): str(v).capitalize() for k, v in config["id_to_accent"].items()}

with open(test_json, "r", encoding="utf-8") as f:
    records = json.load(f)

# STRICT L2-ARCTIC ONLY: drop any LibriSpeech rows.
records = [r for r in records if not is_librispeech_record(r)]
print(f"Test records (L2-ARCTIC only): {len(records)}")

# Resolve paths to local extracted L2-ARCTIC location.
for r in records:
    r["wav_path_resolved"] = resolve_l2arctic_path(r, LOCAL_L2ARCTIC)
    r["transcript"] = str(r["transcript"]).strip().lower()
    r["accent_id"] = int(r["accent_id"])

valid_records = [r for r in records if os.path.isfile(r["wav_path_resolved"])]
print(f"Valid L2-ARCTIC audio files found: {len(valid_records)} / {len(records)}")
assert len(valid_records) > 0, (
    "No L2-ARCTIC audio resolved. Make sure speaker zips are extracted under "
    f"{LOCAL_L2ARCTIC}"
)

# Filter to non-native accents only (Indian, Mandarin, Korean, Arabic, ...).
selection_pool = [
    r for r in valid_records
    if accent_map.get(r["accent_id"], "").strip().lower() != "native"
]
print(f"Non-native L2-ARCTIC records: {len(selection_pool)} / {len(valid_records)}")
assert len(selection_pool) > 0, "No non-native L2-ARCTIC records found."

# Show per-accent counts.
from collections import Counter
acc_counts = Counter(accent_map.get(r["accent_id"], "?") for r in selection_pool)
print("Per-accent pool:", dict(acc_counts))

by_accent: Dict[int, List[dict]] = {}
for r in selection_pool:
    by_accent.setdefault(r["accent_id"], []).append(r)

candidate_rows: List[dict] = []
for aid in sorted(by_accent):
    group = by_accent[aid]
    random.shuffle(group)
    take_n = max(1, CANDIDATE_POOL_PER_ACCENT if SELECTION_MODE == "best_delta" else NUM_CLIPS)
    candidate_rows.extend(group[:take_n])

if SELECTION_MODE == "balanced_random":
    chosen = pick_equally_spaced(candidate_rows, NUM_CLIPS)
else:
    chosen = candidate_rows

print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print("Candidates to evaluate:", len(chosen))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_processor, baseline_model = build_baseline_model(device)
full_processor, full_model = build_full_model(device, ckpt_path)

os.makedirs(OUTPUT_DIR, exist_ok=True)
clips_out_dir = os.path.join(OUTPUT_DIR, "clips")
os.makedirs(clips_out_dir, exist_ok=True)

scored_rows: List[DemoRow] = []
for i, r in enumerate(chosen, start=1):
    wav = r["wav_path_resolved"]
    gt = r["transcript"]
    accent = accent_map.get(r["accent_id"], f"id_{r['accent_id']}")
    clip_id = f"{r.get('speaker', 'spk')}_{i}"

    baseline_pred = transcribe_one(wav, baseline_processor, baseline_model, device)
    full_pred = transcribe_one(wav, full_processor, full_model, device)

    ext = os.path.splitext(wav)[1].lower() or ".wav"
    local_audio_name = f"{clip_id}{ext}"
    local_audio_abs = os.path.join(clips_out_dir, local_audio_name)
    shutil.copy2(wav, local_audio_abs)
    local_audio_rel = os.path.join("clips", local_audio_name).replace("\\", "/")

    row = DemoRow(
        clip_id=clip_id,
        accent=accent,
        audio_file=local_audio_rel,
        ground_truth=gt,
        baseline_pred=baseline_pred,
        full_pred=full_pred,
        baseline_wer=wer([normalize_text(gt)], [normalize_text(baseline_pred)]),
        full_wer=wer([normalize_text(gt)], [normalize_text(full_pred)]),
        baseline_cer=cer([normalize_text(gt)], [normalize_text(baseline_pred)]),
        full_cer=cer([normalize_text(gt)], [normalize_text(full_pred)]),
    )
    scored_rows.append(row)
    print(f"[{i}/{len(chosen)}] {clip_id} ({accent}) WER {row.baseline_wer:.3f}->{row.full_wer:.3f}")

if SELECTION_MODE == "best_delta":
    # Prefer rows with non-trivial improvement (delta > 0) and accent diversity.
    improved = [r for r in scored_rows if r.wer_delta > 1e-6]
    ranked = sorted(improved or scored_rows, key=lambda x: x.wer_delta, reverse=True)
    demo_rows: List[DemoRow] = []
    seen_accents = set()
    for row in ranked:
        if row.accent not in seen_accents and len(demo_rows) < NUM_CLIPS:
            demo_rows.append(row)
            seen_accents.add(row.accent)
    if len(demo_rows) < NUM_CLIPS:
        for row in ranked:
            if row not in demo_rows:
                demo_rows.append(row)
                if len(demo_rows) >= NUM_CLIPS:
                    break
else:
    demo_rows = scored_rows[:NUM_CLIPS]

print("\n=== Demo selection summary ===")
for r in demo_rows:
    print(
        f"- {r.clip_id} ({r.accent}): "
        f"WER {r.baseline_wer:.3f} -> {r.full_wer:.3f} (delta {r.wer_delta:.3f}) | "
        f"CER {r.baseline_cer:.3f} -> {r.full_cer:.3f} (delta {r.cer_delta:.3f})"
    )

if demo_rows:
    avg_b_wer = sum(r.baseline_wer for r in demo_rows) / len(demo_rows)
    avg_f_wer = sum(r.full_wer for r in demo_rows) / len(demo_rows)
    avg_b_cer = sum(r.baseline_cer for r in demo_rows) / len(demo_rows)
    avg_f_cer = sum(r.full_cer for r in demo_rows) / len(demo_rows)
    print(
        f"\nAvg WER: {avg_b_wer:.3f} -> {avg_f_wer:.3f} (delta {avg_b_wer - avg_f_wer:.3f})"
    )
    print(
        f"Avg CER: {avg_b_cer:.3f} -> {avg_f_cer:.3f} (delta {avg_b_cer - avg_f_cer:.3f})"
    )

demo_csv = os.path.join(OUTPUT_DIR, "demo_clips.csv")
with open(demo_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["clip_id", "accent", "audio_file", "ground_truth", "baseline_pred", "full_pred"])
    for r in demo_rows:
        w.writerow([r.clip_id, r.accent, r.audio_file, r.ground_truth, r.baseline_pred, r.full_pred])

metrics_csv = os.path.join(OUTPUT_DIR, "demo_metrics.csv")
with open(metrics_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["clip_id", "accent", "baseline_wer", "full_wer", "wer_delta", "baseline_cer", "full_cer", "cer_delta"])
    for r in demo_rows:
        w.writerow([
            r.clip_id,
            r.accent,
            f"{r.baseline_wer:.6f}",
            f"{r.full_wer:.6f}",
            f"{r.wer_delta:.6f}",
            f"{r.baseline_cer:.6f}",
            f"{r.full_cer:.6f}",
            f"{r.cer_delta:.6f}",
        ])

html_path = os.path.join(OUTPUT_DIR, "demo_showcase.html")
render_html(demo_rows, html_path, TITLE)

print("Final demo clips:", len(demo_rows))
print("Saved:", demo_csv)
print("Saved:", metrics_csv)
print("Saved:", html_path)
print("Copied clips:", clips_out_dir)

Test records (L2-ARCTIC only): 1583
Valid L2-ARCTIC audio files found: 1583 / 1583
Non-native L2-ARCTIC records: 1583 / 1583
Per-accent pool: {'Korean': 335, 'Indian': 457, 'Mandarin': 451, 'Arabic': 340}
Device: cuda
Candidates to evaluate: 16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

[1/16] RRBI_1 (Indian) WER 0.182->0.091
[2/16] SVBI_2 (Indian) WER 0.000->0.000
[3/16] RRBI_3 (Indian) WER 0.143->0.143
[4/16] NJS_4 (Indian) WER 0.571->0.429
[5/16] HQTV_5 (Mandarin) WER 0.556->0.444
[6/16] HQTV_6 (Mandarin) WER 0.273->0.364
[7/16] HQTV_7 (Mandarin) WER 0.308->0.231
[8/16] TXHC_8 (Mandarin) WER 0.231->0.231
[9/16] HKK_9 (Korean) WER 0.375->0.250
[10/16] HKK_10 (Korean) WER 0.429->0.286
[11/16] YDCK_11 (Korean) WER 0.200->0.200
[12/16] YDCK_12 (Korean) WER 0.000->0.000
[13/16] ABA_13 (Arabic) WER 0.500->0.125
[14/16] ABA_14 (Arabic) WER 0.364->0.273
[15/16] ABA_15 (Arabic) WER 0.200->0.100
[16/16] ZHAA_16 (Arabic) WER 0.000->0.000

=== Demo selection summary ===
- ABA_13 (Arabic): WER 0.500 -> 0.125 (delta 0.375) | CER 0.551 -> 0.041 (delta 0.510)
- NJS_4 (Indian): WER 0.571 -> 0.429 (delta 0.143) | CER 0.186 -> 0.163 (delta 0.023)
- HKK_10 (Korean): WER 0.429 -> 0.286 (delta 0.143) | CER 0.102 -> 0.041 (delta 0.061)

Avg WER: 0.500 -> 0.280 (delta 0.220)
Avg CER: 0.28

In [6]:
# Zip artifacts for easy download
import os
import zipfile

zip_path = os.path.join(OUTPUT_DIR, "demo_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for fn in files:
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, OUTPUT_DIR)
            zf.write(full, arcname=rel)
print("Zip ready:", zip_path)
print("HTML:", os.path.join(OUTPUT_DIR, "demo_showcase.html"))

Zip ready: /content/demo_artifacts/demo_artifacts.zip
HTML: /content/demo_artifacts/demo_showcase.html


In [7]:
from google.colab import files
files.download("/content/demo_artifacts/demo_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>